**Role in the paper.** Quality control and gap filling of the station data for one city with the
MetObs-toolkit (v1.1.1), for the urban climate network (`city_network`) and the meteorological service stations
(`meteo_stations`); the pipeline is described in SI Appendix, section 2.
Input: raw third-party data in `data/raw/observations/<City>/` (not redistributed, see README).
Output: cleaned series and station metadata in `data/processed/observations/<City>/<network>/`; the `_gapfree` subset
(stations without remaining gaps) is used in the analysis. Next step: `scripts/preprocessing/clean_obs_to_datasets.ipynb`.

In [ ]:
import metobs_toolkit
import matplotlib.pyplot as plt
import pandas as pd
import sys
from collections import Counter, defaultdict
from pathlib import Path
ROOT = Path.cwd().parent.parent.parent
import numpy as np

# City Network


In [ ]:
city = 'Amsterdam'
network= 'city_network'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"

save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'
tables_dir = ROOT / f"results/tables/{city}/{network}"
plot_dir = ROOT / f"results/figures/{city}/{network}"
save_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)
save_pkl_dir.mkdir(parents=True, exist_ok=True)

## Loading data


In [ ]:
# Define the paths to your files: 
# adapt these paths to your local setup
data_file = data_dir / f'{city}_raw.csv'
meta_data_file = data_dir /f'station_metadata_{city}.csv'
template = data_dir / 'template.json'

# Initiate a dataset:
dataset = metobs_toolkit.Dataset()

# Import your data :
dataset.import_data_from_file(
    input_data_file = data_file,
    input_metadata_file = meta_data_file,
    template_file = template,
    #The following arguments are filled with default values.
    freq_estimation_method = "median",
    freq_estimation_simplify_tolerance = "2min",
    origin_simplify_tolerance = "5min",
    timestamp_tolerance = "4min",
    kwargs_data_read = {},
    kwargs_metadata_read = {},
    templatefile_is_url = False,
)


In [ ]:
dataset.get_info()

In [ ]:
stations = dataset.metadf.index.get_level_values('name').to_list()

In [ ]:
#info from Google earht engine (GEE) for the station locations, see examples for setup and usage. 
dataset.get_LCZ()
dataset.get_altitude()
dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['LCZ'] ,save=True, filepath=f'{plot_dir}/LCZ_map_{city}_{network}', overwrite=True)
dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['worldcover'] ,save=True, filepath=f'{plot_dir}/Worldcover_map_{city}_{network}', overwrite=True)


In [ ]:
dataset.get_station('D2222').site.set_altitude(0)
dataset.get_station('D2226').site.set_altitude(0)
dataset.get_station('D2241').site.set_altitude(0)

dataset.metadf

## Temperature 

### Quality Control


#### Basic QC (gross value, repititions, persistence, spikes, window variation check)

In [ ]:
#first plot of the full dataset
%matplotlib widget
dataset.make_plot(obstype='temp', #Which observation type to plot. (See dataset.present_observations)
                  colorby='station', #if 'station', each station will be a different color
                  show_outliers=True,
                  show_gaps=True)


In [ ]:
#Resampling
dataset.resample(target_freq='1h')

In [ ]:

#1. gross value check
target = 'temp'
dataset.gross_value_check(
            obstype=target,
            lower_threshold=-10.0,
            upper_threshold=40.0)

#2. repetitions check
print(f'Applying repetitions check on {target} observations')
dataset.repetitions_check(
            obstype=target,
            max_N_repetitions=5
)
#3. persistence check
print(f'Applying persistence check on {target} observations')
dataset.persistence_check(
            obstype=target,
            timewindow='5h',
            min_records_per_window=3)


In [ ]:

#4. step check
print(f'Applying step check on {target} observations')
dataset.step_check(
            obstype=target,
            max_increase_per_second = 5.0 / 3600.0, #depends on standard unit! 
            max_decrease_per_second = -5.0 / 3600.0) #depends on standard unit! 

#5. window variation check
print(f'Applying window variation check on {target} observations')
dataset.window_variation_check(
            obstype=target,
            timewindow='4h',
            min_records_per_window=3,
            max_increase_per_second= 5.0 / 3600.0, #depends on standard unit! 
            max_decrease_per_second = -5.0 / 3600.0, #depends on standard unit! 
            )


In [ ]:
dataset.get_qc_stats(obstype='temp',make_plot=True)


In [ ]:
dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_after_basicQC.pkl', overwrite=True)

#### Buddy Check QC

In [ ]:
dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_after_basicQC.pkl')


In [ ]:
#6. buddy check
target = 'temp'
print(f'Applying buddy check on {target} observations')
dataset.buddy_check(
        obstype=target,
        #main check settings
        spatial_buddy_radius=3000, #15km definition of buddy radius
        spatial_z_threshold=2.7, #outlier threshold
        #requirements
        min_sample_size=5,
        max_alt_diff=50,  # Maximum elevation difference between stations
        N_iter=3, #Number of iterations
        instantaneous_tolerance='5min', #Max timestamp tolerance for 'at the same time'
        lapserate=6.5/1000, #Specify the variation with altitude, if None no correction is applied
        min_sample_spread=1.0,  
)



In [ ]:
info_df = dataset.outliersdf.reset_index().groupby('name')['label'].value_counts()
info_df.xs('buddy check outlier', level='label').sort_values(ascending=False)

In [ ]:
dataset.get_qc_stats(obstype='temp',make_plot=True)


In [ ]:
dataset.get_station('D2244').make_plot(obstype='temp', colorby='label', show_outliers=True, show_gaps=True)

In [ ]:
dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_after_buddycheck.pkl', overwrite=True)

### Gap Filling

#### interpolation

In [ ]:
metobs_toolkit.__version__

In [ ]:
dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_after_buddycheck.pkl')
amount_of_outliers_before = len(dataset.outliersdf)
print(f"Number of outliers before qc->gaps: {amount_of_outliers_before}")

In [ ]:
dataset.convert_outliers_to_gaps(obstype='temp')

In [ ]:
#apply time(=linear but dt aware) interpolation for small gaps (up to 2 consecutive missing values)
dataset.interpolate_gaps(
                    obstype='temp', #Which gaps to fill
                    overwrite_fill = False, #Overwrite previous filled values if they are present
                    method='time', #which interpolation method
                    max_gap_duration_to_fill=pd.Timedelta("4h")
                    )


In [ ]:

#apply polynomial interpolation for larger gaps (up to 5 consecutive missing values)
dataset.interpolate_gaps(
                obstype='temp',
                method='polynomial',
                overwrite_fill=False, #the gaps that are already filled are skipped (i.g. chainig GF methods)
                n_leading_anchors=3, #at least 3 leading anchors are needed for 3th order polynomial interpolation
                n_trailing_anchors=3, #at least 3 trailing anchors are needed for 3th order polynomial interpolation
                max_gap_duration_to_fill=pd.Timedelta("8h"), #the maximum number of consecutive timestamps to fill. The result is dependent on the time-resolution of the gap (=equal to that of the related SensorData). Defaults to 10
                max_lead_to_gap_distance='24h', #the maximum distance (in time) beween the leading anchors and the start of the gap.
                max_trail_to_gap_distance = '24h',
                method_kwargs={'order':3}, #all extra arguments to pass to the pandas.Dataframe.interpolate method.
)


In [ ]:
dataset.get_station('D2244').make_plot(obstype='temp', colorby='label', show_outliers=True, show_gaps=True)


In [ ]:
dataset.make_plot( obstype='temp', colorby='station', show_outliers=True, show_gaps=False)


In [ ]:
dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_after_interpolation_gf.pkl', overwrite=True) #set overwrite to True to overwrite if the file already exists, otherwise an error is raised.

#### fill gaps with model data

In [ ]:
dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_after_interpolation_gf.pkl')

Remaining gaps are filled with ERA5-Land, debiased with the station's own observations around each gap (MetObs-toolkit).

In [ ]:
era5_manager = metobs_toolkit.default_GEE_datasets['ERA5-land']

ERA5_data = f"{data_dir}\\ERA5-land_timeseries_data_{city}.csv"
#I added the file with the ERA5 data to the datafolder so you should just adapt the path to your dir, but try to download yourself based on the example scripts

dataset.import_gee_data_from_file( filepath= ERA5_data,  gee_dynamic_manager=era5_manager  )


In [ ]:
dataset.fill_gaps_with_weighted_diurnal_debiased_modeldata(
            obstype='temp',
            leading_period_duration=pd.Timedelta('10d'),
            trailing_period_duration=pd.Timedelta('10d'),
            min_lead_debias_sample_size=2,
            min_trail_debias_sample_size=2,
            max_gap_duration_to_fill=pd.Timedelta('33d'))


In [ ]:
gaps = dataset.gap_overview_df()
remaining_gaps = gaps[gaps['label']=='failed gapfill']
remaining_gaps

In [ ]:
dataset.fill_gaps_with_diurnal_debiased_modeldata(
            obstype='temp',
            overwrite_fill=False,
            leading_period_duration=pd.Timedelta('10d'),
            trailing_period_duration=pd.Timedelta('10d'),
            min_debias_sample_size=2,
            max_gap_duration_to_fill=pd.Timedelta('33d'))

In [ ]:
dataset.fill_gaps_with_debiased_modeldata(
            obstype='temp',
            overwrite_fill=False,
            leading_period_duration=pd.Timedelta('10d'),
            trailing_period_duration=pd.Timedelta('10d'),
            min_leading_records_total=5,
            min_trailing_records_total=5,
            max_gap_duration_to_fill=pd.Timedelta('30d'))

In [ ]:
dataset.fill_gaps_with_debiased_modeldata(
            obstype='temp',
            overwrite_fill=False,
            leading_period_duration=pd.Timedelta('1h'),
            trailing_period_duration=pd.Timedelta('7d'),
            min_leading_records_total=0,
            min_trailing_records_total=5,
            max_gap_duration_to_fill=pd.Timedelta('6d'))
dataset.fill_gaps_with_debiased_modeldata(
            obstype='temp',
            overwrite_fill=False,
            leading_period_duration=pd.Timedelta('7d'),
            trailing_period_duration=pd.Timedelta('1h'),
            min_leading_records_total=5,
            min_trailing_records_total=0,
            max_gap_duration_to_fill=pd.Timedelta('6d'))

In [ ]:
gaps = dataset.gap_overview_df()
remaining_gaps = gaps[gaps['label']=='failed gapfill']
remaining_gaps

In [ ]:
stations = dataset.metadf.index.get_level_values('name').to_list()
for station in stations:
    dataset.get_station(station).make_plot(obstype='temp', colorby='label', show_outliers=True, show_gaps=True)

In [ ]:
stations_without_gapsin_2018_2020 =['D2195', 'D2198', 'D2199', 'D2221', 'D2223', 'D2225', 'D2229', 'D2230', 'D2231', 'D2235', 'D2236', 'D2239', 'D2240', 'D2241', 'D2244', 'D2246']

In [ ]:
%matplotlib widget
dataset.subset_by_stations(stationnames=stations_without_gapsin_2018_2020).make_plot(obstype='temp', #Which observation type to plot. (See dataset.present_observations)
                  colorby='label', #if 'station', each station will be a different color
                  show_outliers=True,
                  show_gaps=True)

#### Save final products

In [ ]:
dataset.subset_by_stations(stationnames=stations_without_gapsin_2018_2020).save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned_gapfree2018_2020.pkl', overwrite=True) #only save the subset of the dataset without unfilled gaps, to have a clean dataset for the next steps of the analysis.
dataset.subset_by_stations(stationnames=stations_without_gapsin_2018_2020).to_csv(f'{save_dir}\\{city}_{network}_cleaned_gapfree2018_2020.csv', overwrite=True)

dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned_total.pkl', overwrite=True) #only save the subset of the dataset without unfilled gaps, to have a clean dataset for the next steps of the analysis.
dataset.to_csv(f'{save_dir}\\{city}_{network}_cleaned_total.csv', overwrite=True)

dataset.metadf.to_csv(f'{save_dir}\\{city}_{network}_metadata.csv')

# Meteo data

In [ ]:
city = 'Amsterdam'
network= 'meteo_stations'
data_dir = ROOT / f"data/raw/observations/{city}/{network}"

save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'
tables_dir = ROOT / f"results/tables/{city}/{network}"
plot_dir = ROOT / f"results/figures/{city}/{network}"
save_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)
save_pkl_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# Define the paths to your files: 
# adapt these paths to your local setup
data_file = data_dir / f'{city}_meteo_data.csv'
meta_data_file = data_dir /f'{city}_meteo_metadata.csv'
template = data_dir / 'template.json'

# Initiate a dataset:
meteo_dataset = metobs_toolkit.Dataset()

# Import your data :
meteo_dataset.import_data_from_file(
    input_data_file = data_file,
    input_metadata_file = meta_data_file,
    template_file = template,
    #The following arguments are filled with default values.
    freq_estimation_method = "median",
    freq_estimation_simplify_tolerance = "2min",
    origin_simplify_tolerance = "5min",
    timestamp_tolerance = "4min",
    kwargs_data_read = {},
    kwargs_metadata_read = {},
    templatefile_is_url = False,
)

In [ ]:
meteo_dataset.get_info()

In [ ]:
meteo_dataset.gap_overview_df()

In [ ]:
#info from Google earht engine (GEE) for the station locations, see examples for setup and usage. 
meteo_dataset.get_LCZ()
meteo_dataset.get_altitude()
meteo_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['LCZ'] ,save=True, filepath=f'{plot_dir}/LCZ_map_{city}_{network}', overwrite=True)
meteo_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['worldcover'] ,save=True, filepath=f'{plot_dir}/Worldcover_map_{city}_{network}', overwrite=True)


In [ ]:
meteo_dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned.pkl', overwrite=True) #set overwrite to True to overwrite if the file already exists, otherwise an error is raised.
meteo_dataset.to_csv(f'{save_dir}\\{city}_{network}_cleaned.csv', overwrite=True)
meteo_dataset.metadf.to_csv(f'{save_dir}\\{city}_{network}_metadata.csv')

# Combine meteo and city network datasets

In [ ]:
subset = 'gapfree2018_2020'

In [ ]:
city = 'Amsterdam'
network= 'city_network'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"
save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'

dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned_{subset}.pkl')

In [ ]:
city = 'Amsterdam'
network= 'meteo_stations'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"
save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'

meteo_dataset = metobs_toolkit.import_dataset_from_pkl(f'{save_pkl_dir}\\{city}_meteo_stations_cleaned.pkl')

In [ ]:
city = 'Amsterdam'
network= 'combined_network'

data_dir = ROOT / f"data/raw/observations/{city}/{network}"

save_dir = ROOT / f"data/processed/observations/{city}/{network}"
save_pkl_dir = save_dir / 'pkl'
tables_dir = ROOT / f"results/tables/{city}/{network}"
plot_dir = ROOT / f"results/figures/{city}/{network}"
save_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)
plot_dir.mkdir(parents=True, exist_ok=True)
save_pkl_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
combi_dataset = dataset+meteo_dataset

In [ ]:
combi_dataset.metadf

In [ ]:
#info from Google earht engine (GEE) for the station locations, see examples for setup and usage. 
combi_dataset.get_LCZ()
combi_dataset.get_altitude()
combi_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['LCZ'] ,save=True, filepath=f'{plot_dir}/LCZ_map_{city}_{network}_{subset}', overwrite=True)
combi_dataset.make_gee_plot(gee_manager= metobs_toolkit.default_GEE_datasets['worldcover'] ,save=True, filepath=f'{plot_dir}/Worldcover_map_{city}_{network}_{subset}', overwrite=True)


In [ ]:
combi_dataset.save_dataset_to_pkl(f'{save_pkl_dir}\\{city}_{network}_cleaned_{subset}.pkl', overwrite=True) #set overwrite to True to overwrite if the file already exists, otherwise an error is raised.
combi_dataset.to_csv(f'{save_dir}\\{city}_{network}_cleaned_{subset}.csv', overwrite=True)
combi_dataset.metadf.to_csv(f'{save_dir}\\{city}_{network}_{subset}_metadata.csv')

In [ ]:
table_save_dir = ROOT / f"results/tables/{city}"
table_save_dir.mkdir(parents=True, exist_ok=True)
wc = combi_dataset.get_landcover_fractions(buffers = [10,50,100,200], aggregate=False)
wc = wc.replace(np.nan,0)
wc['rural_fraction'] = wc['Grassland'] + wc['Cropland'] + wc['Bare / sparse vegetation'] + wc['Tree cover']
wc['urban_fraction'] = wc['Built-up'] 
wc.to_csv(f'{table_save_dir}/landcover_fractions_{city}_{network}_{subset}.csv', index=True)

In [ ]:
wc